In [1]:
import os
import random
import pathlib
import warnings

import numpy as np
import pandas as pd

from math import ceil

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings("ignore")

ENABLE_TF = os.environ.get("ENABLE_TF", "0").strip().lower() in (
    "1",
    "true",
    "yes",
    "y",
)

TF_OK = False
tf = None
K = None
TF_IMPORT_ERROR = None
if ENABLE_TF:
    try:
        import tensorflow as _tf  # noqa: F401
        from tensorflow import keras as _K  # noqa: F401

        tf = _tf
        K = _K
        tf.random.set_seed(SEED)
        TF_OK = True
    except Exception as e:
        TF_OK = False
        TF_IMPORT_ERROR = repr(e)
else:
    TF_OK = False
    TF_IMPORT_ERROR = "ENABLE_TF=0 (forced tabular-only fallback)"

USE_DICOM = os.environ.get("USE_DICOM", "0").strip().lower() in (
    "1",
    "true",
    "yes",
    "y",
)

PYDICOM_OK = False
PYDICOM_IMPORT_ERROR = None
pydicom = None
if USE_DICOM:
    try:
        import pydicom as _pydicom  # noqa: F401

        pydicom = _pydicom
        PYDICOM_OK = True
    except Exception as e:
        PYDICOM_OK = False
        PYDICOM_IMPORT_ERROR = repr(e)
else:
    PYDICOM_OK = False
    PYDICOM_IMPORT_ERROR = "USE_DICOM=0 (forced tabular-only fallback)"

print("ENABLE_TF:", ENABLE_TF)
print("TF_OK:", TF_OK)
if not TF_OK:
    print(
        "TensorFlow unavailable; will use tabular-only fallback. Info:", TF_IMPORT_ERROR
    )

print("USE_DICOM:", USE_DICOM)
print("PYDICOM_OK:", PYDICOM_OK)
if not PYDICOM_OK:
    print(
        "DICOM path disabled/unavailable; will use tabular-only fallback. Info:",
        PYDICOM_IMPORT_ERROR,
    )



ENABLE_TF: False
TF_OK: False
TensorFlow unavailable; will use tabular-only fallback. Info: ENABLE_TF=0 (forced tabular-only fallback)
USE_DICOM: False
PYDICOM_OK: False
DICOM path disabled/unavailable; will use tabular-only fallback. Info: USE_DICOM=0 (forced tabular-only fallback)


In [2]:
raw_test = pd.read_csv("/kaggle/input/osic-pulmonary-fibrosis-progression/test.csv")
raw_train = pd.read_csv("/kaggle/input/osic-pulmonary-fibrosis-progression/train.csv")
X_prediction = pd.read_csv(
    "/kaggle/input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)



In [3]:
TEST_PATH = "/kaggle/input/osic-pulmonary-fibrosis-progression/test"

DESIRED_SIZE = (30, 256, 256)
BATCH_SIZE = 256

clip_bounds = (-1000, 200)
pre_calculated_mean = 0.02865046213070556



In [4]:
X_prediction["Patient"] = X_prediction["Patient_Week"].str.extract(r"(.*)_.*")
X_prediction["Weeks"] = X_prediction["Patient_Week"].str.extract(r".*_(.*)").astype(int)
X_prediction = X_prediction[["Patient", "Weeks", "Patient_Week"]]

test_base = (
    raw_test.sort_values(["Patient", "Weeks"])
    .groupby("Patient", as_index=False)
    .first()
    .rename(columns={"Weeks": "Min_week", "FVC": "Base_FVC"})
)

X_prediction = X_prediction.merge(
    test_base[
        ["Patient", "Min_week", "Base_FVC", "Percent", "Age", "Sex", "SmokingStatus"]
    ],
    how="left",
    on="Patient",
)

X_prediction = X_prediction[
    [
        "Patient",
        "Min_week",
        "Base_FVC",
        "Percent",
        "Age",
        "Sex",
        "SmokingStatus",
        "Weeks",
        "Patient_Week",
    ]
].reset_index(drop=True)

X_prediction = X_prediction.merge(
    test_base[["Patient", "Min_week"]].rename(columns={"Min_week": "Anchor_week"}),
    on="Patient",
    how="left",
)



In [5]:
X_prediction["Base_week"] = X_prediction["Weeks"] - X_prediction["Anchor_week"]



In [6]:
from sklearn.preprocessing import OneHotEncoder as SklearnOneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.exceptions import NotFittedError


class OneHotEncoder(SklearnOneHotEncoder):
    def __init__(self, **kwargs):
        super(OneHotEncoder, self).__init__(**kwargs)
        self.fit_flag = False

    def fit(self, X, **kwargs):
        out = super().fit(X)
        self.fit_flag = True
        return out

    def transform(self, X, categories, index="", name="", **kwargs):
        sparse_matrix = super(OneHotEncoder, self).transform(X)
        new_columns = self.get_new_columns(X=X, name=name, categories=categories)
        d_out = pd.DataFrame(sparse_matrix.toarray(), columns=new_columns, index=index)
        return d_out

    def fit_transform(self, X, categories, index, name, **kwargs):
        self.fit(X)
        return self.transform(X, categories=categories, index=index, name=name)

    def get_new_columns(self, X, name, categories):
        new_columns = []
        for j in range(len(categories)):
            new_columns.append("{}_{}".format(name, categories[j]))
        return new_columns


def standardisation(x, u, s):
    return (x - u) / s


def normalization(x, ma, mi):
    denom = (ma - mi) if (ma - mi) != 0 else 1.0
    return (x - mi) / denom


class data_preparation:
    def __init__(self, bool_normalization=True, bool_standard=False):
        self.enc_sex = LabelEncoder()
        self.enc_smok = LabelEncoder()
        try:
            self.onehotenc_smok = OneHotEncoder(
                sparse_output=True, handle_unknown="ignore"
            )
        except TypeError:
            self.onehotenc_smok = OneHotEncoder(sparse=True, handle_unknown="ignore")
        self.standardisation = bool_standard
        self.normalization = bool_normalization

    def __call__(self, data_untransformed):
        data = data_untransformed.copy(deep=True)

        try:
            data["Sex"] = self.enc_sex.transform(data["Sex"].values)
            data["SmokingStatus"] = self.enc_smok.transform(
                data["SmokingStatus"].values
            )

            data = pd.concat(
                [
                    data.drop(columns=["SmokingStatus"]),
                    self.onehotenc_smok.transform(
                        data["SmokingStatus"].values.reshape(-1, 1),
                        categories=self.enc_smok.classes_,
                        name="",
                        index=data.index,
                    ).astype(int),
                ],
                axis=1,
            )

            if self.standardisation:
                data["Base_week"] = standardisation(
                    data["Base_week"], self.base_week_mean, self.base_week_std
                )
                data["Base_FVC"] = standardisation(
                    data["Base_FVC"], self.base_fvc_mean, self.base_fvc_std
                )
                data["Percent"] = standardisation(
                    data["Percent"], self.base_percent_mean, self.base_percent_std
                )
                data["Age"] = standardisation(data["Age"], self.age_mean, self.age_std)
                data["Weeks"] = standardisation(
                    data["Weeks"], self.weeks_mean, self.weeks_std
                )

            if self.normalization:
                data["Base_week"] = normalization(
                    data["Base_week"], self.base_week_max, self.base_week_min
                )
                data["Base_FVC"] = normalization(
                    data["Base_FVC"], self.base_fvc_max, self.base_fvc_min
                )
                data["Percent"] = normalization(
                    data["Percent"], self.base_percent_max, self.base_percent_min
                )
                data["Age"] = normalization(data["Age"], self.age_max, self.age_min)
                data["Weeks"] = normalization(
                    data["Weeks"], self.weeks_max, self.weeks_min
                )
                data["Min_week"] = normalization(
                    data["Min_week"], self.min_week_max, self.min_week_min
                )

        except NotFittedError:
            data["Sex"] = self.enc_sex.fit_transform(data["Sex"].values)
            data["SmokingStatus"] = self.enc_smok.fit_transform(
                data["SmokingStatus"].values
            )

            data = pd.concat(
                [
                    data.drop(columns=["SmokingStatus"]),
                    self.onehotenc_smok.fit_transform(
                        data["SmokingStatus"].values.reshape(-1, 1),
                        categories=self.enc_smok.classes_,
                        name="",
                        index=data.index,
                    ).astype(int),
                ],
                axis=1,
            )

            if self.standardisation:
                self.base_week_mean = data["Base_week"].mean()
                self.base_week_std = data["Base_week"].std()
                data["Base_week"] = standardisation(
                    data["Base_week"], self.base_week_mean, self.base_week_std
                )

                self.base_fvc_mean = data["Base_FVC"].mean()
                self.base_fvc_std = data["Base_FVC"].std()
                data["Base_FVC"] = standardisation(
                    data["Base_FVC"], self.base_fvc_mean, self.base_fvc_std
                )

                self.base_percent_mean = data["Percent"].mean()
                self.base_percent_std = data["Percent"].std()
                data["Percent"] = standardisation(
                    data["Percent"], self.base_percent_mean, self.base_percent_std
                )

                self.age_mean = data["Age"].mean()
                self.age_std = data["Age"].std()
                data["Age"] = standardisation(data["Age"], self.age_mean, self.age_std)

                self.weeks_mean = data["Weeks"].mean()
                self.weeks_std = data["Weeks"].std()
                data["Weeks"] = standardisation(
                    data["Weeks"], self.weeks_mean, self.weeks_std
                )

            if self.normalization:
                self.base_week_min = data["Base_week"].min()
                self.base_week_max = data["Base_week"].max()
                data["Base_week"] = normalization(
                    data["Base_week"], self.base_week_max, self.base_week_min
                )

                self.base_fvc_min = data["Base_FVC"].min()
                self.base_fvc_max = data["Base_FVC"].max()
                data["Base_FVC"] = normalization(
                    data["Base_FVC"], self.base_fvc_max, self.base_fvc_min
                )

                self.base_percent_min = data["Percent"].min()
                self.base_percent_max = data["Percent"].max()
                data["Percent"] = normalization(
                    data["Percent"], self.base_percent_max, self.base_percent_min
                )

                self.age_min = data["Age"].min()
                self.age_max = data["Age"].max()
                data["Age"] = normalization(data["Age"], self.age_max, self.age_min)

                self.weeks_min = data["Weeks"].min()
                self.weeks_max = data["Weeks"].max()
                data["Weeks"] = normalization(
                    data["Weeks"], self.weeks_max, self.weeks_min
                )

                self.min_week_min = data["Min_week"].min()
                self.min_week_max = data["Min_week"].max()
                data["Min_week"] = normalization(
                    data["Min_week"], self.min_week_max, self.min_week_min
                )

        return data




In [7]:
train_for_prep = raw_train.copy()
train_for_prep = train_for_prep.rename(columns={"Weeks": "Min_week", "FVC": "Base_FVC"})
train_for_prep["Weeks"] = train_for_prep["Min_week"]  # keep required column present
train_for_prep["Base_week"] = 0

data_prep = data_preparation(bool_normalization=True, bool_standard=False)
_ = data_prep(
    train_for_prep[
        [
            "Patient",
            "Min_week",
            "Base_FVC",
            "Percent",
            "Age",
            "Sex",
            "SmokingStatus",
            "Weeks",
            "Base_week",
        ]
    ]
)

_anchor_cols = X_prediction[["Patient", "Anchor_week", "Base_FVC"]].copy()

X_prediction = data_prep(X_prediction).sort_values("Patient").reset_index(drop=True)
X_prediction = X_prediction.merge(
    _anchor_cols, on="Patient", how="left", suffixes=("", "_raw")
)

# Change (score correctness): ensure we anchor predictions to the *raw* test baseline FVC.
X_prediction["Base_FVC"] = X_prediction["Base_FVC_raw"].astype(np.float32)
X_prediction = X_prediction.drop(columns=["Base_FVC_raw"])



In [8]:
if PYDICOM_OK:
    from scipy.ndimage import zoom
    import scipy.ndimage as ndimage
    from skimage import measure, morphology, segmentation

    class ConvertToHU:
        def __call__(self, imgs, dicom):
            intercept = dicom.RescaleIntercept
            slope = dicom.RescaleSlope
            imgs = (np.array(imgs.to_list()) * slope + intercept).astype(np.int16)
            return imgs

    convertohu = ConvertToHU()

    class Clip:
        def __init__(self, bounds=(-1000, 500)):
            self.min = min(bounds)
            self.max = max(bounds)

        def __call__(self, image):
            image = image.copy()
            image[image < self.min] = self.min
            image[image > self.max] = self.max
            return image

    clip = Clip(clip_bounds)

    class MaskWatershed:
        def __init__(self, min_hu, iterations):
            self.min_hu = min_hu
            self.iterations = iterations

        def __call__(self, image, dicom):
            stack = []
            for slice_idx in range(image.shape[0]):
                sliced = image[slice_idx]
                stack.append(self.seperate_lungs(sliced, self.min_hu, self.iterations))
            return np.stack(stack)

        @staticmethod
        def seperate_lungs(image, min_hu, iterations):
            h, w = image.shape[0], image.shape[1]

            marker_internal, marker_external, marker_watershed = (
                MaskWatershed.generate_markers(image)
            )

            sobel_filtered_dx = ndimage.sobel(image, 1)
            sobel_filtered_dy = ndimage.sobel(image, 0)
            sobel_gradient = np.hypot(sobel_filtered_dx, sobel_filtered_dy)
            maxg = np.max(sobel_gradient)
            if maxg != 0:
                sobel_gradient *= 255.0 / maxg

            try:
                watershed = morphology.watershed(sobel_gradient, marker_watershed)
            except Exception:
                from skimage.segmentation import watershed as seg_watershed

                watershed = seg_watershed(sobel_gradient, marker_watershed)

            outline = ndimage.morphological_gradient(watershed, size=(3, 3))
            outline = outline.astype(bool)

            blackhat_struct = [
                [0, 0, 1, 1, 1, 0, 0],
                [0, 1, 1, 1, 1, 1, 0],
                [1, 1, 1, 1, 1, 1, 1],
                [1, 1, 1, 1, 1, 1, 1],
                [1, 1, 1, 1, 1, 1, 1],
                [0, 1, 1, 1, 1, 1, 0],
                [0, 0, 1, 1, 1, 0, 0],
            ]

            blackhat_struct = ndimage.iterate_structure(blackhat_struct, iterations)
            outline += ndimage.black_tophat(outline, structure=blackhat_struct)

            lungfilter = np.bitwise_or(marker_internal, outline)
            lungfilter = ndimage.morphology.binary_closing(
                lungfilter, structure=np.ones((5, 5)), iterations=3
            )

            segmented = np.where(lungfilter == 1, image, min_hu * np.ones((h, w)))
            return segmented

        @staticmethod
        def generate_markers(image, threshold=-400):
            h, w = image.shape[0], image.shape[1]

            marker_internal = image < threshold
            marker_internal = segmentation.clear_border(marker_internal)
            marker_internal_labels = measure.label(marker_internal)

            areas = [r.area for r in measure.regionprops(marker_internal_labels)]
            areas.sort()

            if len(areas) > 2:
                for region in measure.regionprops(marker_internal_labels):
                    if region.area < areas[-2]:
                        for coordinates in region.coords:
                            marker_internal_labels[coordinates[0], coordinates[1]] = 0

            marker_internal = marker_internal_labels > 0

            external_a = ndimage.binary_dilation(marker_internal, iterations=10)
            external_b = ndimage.binary_dilation(marker_internal, iterations=55)
            marker_external = external_b ^ external_a

            marker_watershed = np.zeros((h, w), dtype=int)
            marker_watershed += marker_internal.astype(int) * 255
            marker_watershed += marker_external.astype(int) * 128

            return marker_internal, marker_external, marker_watershed

    maskwatershed = MaskWatershed(min_hu=min(clip_bounds), iterations=2)

    class Normalize:
        def __init__(self, bounds=(-1000, 500)):
            self.min = min(bounds)
            self.max = max(bounds)

        def __call__(self, image):
            image = image.astype(np.float32)
            image = (image - self.min) / (self.max - self.min)
            return image

    class ZeroCenter:
        def __init__(self, pre_calculated_mean):
            self.pre_calculated_mean = pre_calculated_mean

        def __call__(self, image):
            return image - self.pre_calculated_mean

    normalize = Normalize(bounds=clip_bounds)
    zerocenter = ZeroCenter(pre_calculated_mean=pre_calculated_mean)



In [9]:
if PYDICOM_OK and TF_OK:

    def sort_function(x):
        return int(str(x).split(".")[0])

    def _read(path, patients=[], desired_size=(60, 512, 512)):
        X = np.empty(
            np.concatenate(([len(patients), 1], np.array(desired_size))),
            dtype=np.float32,
        )
        i = 0
        for patient in patients:
            patient_dir = os.path.join(path, patient)
            files = os.listdir(patient_dir)
            dicom = pydicom.dcmread(os.path.join(patient_dir, files[0]))

            list_patient_files = sorted(files, key=sort_function)
            df_paths = [os.path.join(patient_dir, f) for f in list_patient_files]

            df_imgs = convertohu(
                pd.Series(df_paths).apply(lambda x: pydicom.dcmread(x).pixel_array),
                dicom,
            )

            df_imgs = zoom(
                df_imgs,
                np.array(DESIRED_SIZE) / np.array(df_imgs.shape),
                mode="nearest",
            )
            X[i, 0, :, :, :] = zerocenter(
                normalize(maskwatershed(clip(df_imgs), dicom))
            )
            i += 1
        return X

    class DataGenerator(K.utils.Sequence):
        def on_epoch_end(self):
            self.indices = np.arange(len(self.list_IDs))

        def __len__(self):
            return int(ceil(len(self.indices) / self.batch_size))

        def __init__(
            self,
            train,
            list_IDs,
            batch_size=1,
            desired_size=(10, 512, 512),
            img_path=TEST_PATH,
            *args,
            **kwargs,
        ):
            self.train = train
            self.list_IDs = list_IDs
            self.batch_size = batch_size
            self.desired_size = desired_size
            self.img_path = img_path
            self.on_epoch_end()

        def __getitem__(self, index):
            indices = self.indices[
                index * self.batch_size : (index + 1) * self.batch_size
            ]
            list_IDs_temp = [self.list_IDs[k] for k in indices]

            patients = self.train.loc[list_IDs_temp, "Patient"].unique()
            imgs = _read(
                self.img_path, patients=patients, desired_size=self.desired_size
            )

            X1 = self.train.loc[list_IDs_temp, :].reset_index(drop=True)
            X2 = np.transpose(imgs, (0, 2, 3, 4, 1))
            return X1, X2




In [10]:
MODEL_DIR = "/kaggle/input/3d-cnn-mlp/model_30"
CNN_DIR = "/kaggle/input/3d-cnn-mlp/CNN_30"


def _resolve_savedmodel_dir(path):
    path = str(path)
    if os.path.isdir(path) and (
        os.path.exists(os.path.join(path, "saved_model.pb"))
        or os.path.exists(os.path.join(path, "saved_model.pbtxt"))
    ):
        return path

    if not os.path.isdir(path):
        parent = os.path.dirname(path)
    else:
        parent = path

    for root, dirs, files in os.walk(parent):
        if "saved_model.pb" in files or "saved_model.pbtxt" in files:
            return root
        rel_depth = pathlib.Path(root).relative_to(parent).parts
        if len(rel_depth) >= 4:
            dirs[:] = []

    raise OSError(
        f"SavedModel file does not exist at: {path} (and could not resolve under {parent})"
    )


def _make_tfsm_layer(path):
    resolved = _resolve_savedmodel_dir(path)
    for endpoint in ["serving_default", "serve", "call"]:
        try:
            return K.layers.TFSMLayer(resolved, call_endpoint=endpoint)
        except Exception:
            pass
    loaded = tf.saved_model.load(resolved)
    sigs = list(loaded.signatures.keys())
    if not sigs:
        raise ValueError(f"No signatures found in SavedModel at {resolved}")
    return K.layers.TFSMLayer(resolved, call_endpoint=sigs[0])


MODEL_OK = False
model_layer = None
cnn_layer = None
if TF_OK:
    try:
        model_layer = _make_tfsm_layer(MODEL_DIR)
        cnn_layer = _make_tfsm_layer(CNN_DIR)
        MODEL_OK = True
        print("Resolved MODEL_DIR to:", _resolve_savedmodel_dir(MODEL_DIR))
        print("Resolved CNN_DIR to:", _resolve_savedmodel_dir(CNN_DIR))
    except Exception as e:
        MODEL_OK = False
        print(
            "Pretrained SavedModels not available; will use tabular-only fallback. Error:",
            repr(e),
        )
else:
    print("Skipping SavedModel load because TensorFlow is not available.")



Skipping SavedModel load because TensorFlow is not available.


In [11]:
SELECTED_COLUMNS = [
    "Weeks",
    "Percent",
    "Age",
    "Sex",
    "Min_week",
    "Base_FVC",
    "Base_week",
    "_Currently smokes",
    "_Ex-smoker",
    "_Never smoked",
]
for c in SELECTED_COLUMNS:
    if c not in X_prediction.columns:
        X_prediction[c] = 0

if PYDICOM_OK and TF_OK:
    pred_generator = DataGenerator(
        X_prediction,
        X_prediction.index.tolist(),
        batch_size=BATCH_SIZE,
        desired_size=DESIRED_SIZE,
        img_path=TEST_PATH,
    )




In [12]:
def _fit_patient_slopes(train_df: pd.DataFrame):
    """
    Keep the same per-patient linear slope idea, with least-squares fit FVC ~ a + b*Weeks.
    """
    df = train_df[["Patient", "Weeks", "FVC"]].dropna().copy()
    slopes = {}
    counts = {}
    for pid, g in df.groupby("Patient"):
        g = g.sort_values("Weeks")
        x = g["Weeks"].values.astype(np.float64)
        y = g["FVC"].values.astype(np.float64)
        n = int(len(g))
        counts[pid] = n
        if n >= 2 and np.std(x) > 0:
            xm = x.mean()
            ym = y.mean()
            vx = np.mean((x - xm) ** 2)
            if vx > 0:
                slope = float(np.mean((x - xm) * (y - ym)) / vx)
            else:
                slope = np.nan
            slopes[pid] = slope
        else:
            slopes[pid] = np.nan

    slopes = pd.Series(slopes, name="slope").astype(np.float32)
    counts = pd.Series(counts, name="n").astype(np.int32)

    global_slope = float(np.nanmedian(slopes.values))
    if not np.isfinite(global_slope):
        global_slope = 0.0
    slopes = slopes.fillna(global_slope)

    return slopes, counts, global_slope


patient_slopes, patient_counts, global_slope = _fit_patient_slopes(raw_train)
print("Global slope (ml/week):", global_slope)


def _estimate_residual_laplace_scale_anchored(
    train_df: pd.DataFrame,
    slopes: pd.Series,
    counts: pd.Series,
    global_slope_val: float,
) -> float:
    """
    Change (score-aligned, minimal): estimate residual scale using the *same anchoring logic*
    as the submission (patient's first observed week as anchor), then take Laplace MLE scale b = mean(|resid|).
    This better calibrates Confidence for the Laplace log-likelihood than using median(|resid|) or a mismatched anchor.
    """
    df = train_df[["Patient", "Weeks", "FVC"]].dropna().copy()
    anchor = (
        df.sort_values(["Patient", "Weeks"]).groupby("Patient").first().reset_index()
    )
    anchor = anchor.rename(columns={"Weeks": "Anchor_week", "FVC": "Base_FVC"})
    df = df.merge(
        anchor[["Patient", "Anchor_week", "Base_FVC"]], on="Patient", how="left"
    )

    n = df["Patient"].map(counts).fillna(0).astype(np.float32)
    raw_slope = df["Patient"].map(slopes).fillna(global_slope_val).astype(np.float32)

    alpha = (n / (n + 3.0)).astype(np.float32)
    slope = alpha * raw_slope + (1.0 - alpha) * np.float32(global_slope_val)

    # Change (stability toward better score): gentle slope shrinkage to reduce rare huge errors
    # without changing the core linear model (still a slope per patient).
    slope = 0.95 * slope

    slope = np.clip(slope.values.astype(np.float32), -200.0, 200.0)

    pred = df["Base_FVC"].values.astype(np.float32) + slope * (
        df["Weeks"].values.astype(np.float32)
        - df["Anchor_week"].values.astype(np.float32)
    )
    resid = (df["FVC"].values.astype(np.float32) - pred).astype(np.float32)

    b = float(np.mean(np.abs(resid))) if len(resid) else 250.0
    if not np.isfinite(b) or b <= 0:
        b = 250.0
    return float(b)


laplace_b = _estimate_residual_laplace_scale_anchored(
    raw_train, patient_slopes, patient_counts, global_slope
)
print("Estimated Laplace b = mean(|residual|) (ml):", laplace_b)


def _build_tabular_fallback_submission(pred_df: pd.DataFrame) -> pd.DataFrame:
    """
    Same core linear model: FVC = Base_FVC + slope * (Weeks - Anchor_week).

    Changes (score-aligned but minimal):
    - Use slope shrinkage (0.95x) to reduce overconfident extreme slopes.
    - Use constant Confidence = clipped Laplace scale (no horizon growth), since the metric penalizes log(sigma).
    """
    df = pred_df.copy()

    n = df["Patient"].map(patient_counts).fillna(0).astype(np.float32)
    raw_slope = (
        df["Patient"].map(patient_slopes).fillna(global_slope).astype(np.float32)
    )

    alpha = (n / (n + 3.0)).astype(np.float32)
    df["slope"] = alpha * raw_slope + (1.0 - alpha) * np.float32(global_slope)

    # Change (score): gentle shrinkage to reduce occasional large Delta that hurt the LL.
    df["slope"] = 0.95 * df["slope"].astype(np.float32)

    df["slope"] = np.clip(df["slope"].values.astype(np.float32), -200.0, 200.0)

    if "Anchor_week" not in df.columns:
        df = df.merge(
            raw_test.sort_values(["Patient", "Weeks"])
            .groupby("Patient", as_index=False)
            .first()[["Patient", "Weeks"]]
            .rename(columns={"Weeks": "Anchor_week"}),
            on="Patient",
            how="left",
        )
    df["Anchor_week"] = df["Anchor_week"].astype(np.float32)

    anchor_week = df["Anchor_week"].values.astype(np.float32)

    df["FVC_pred"] = df["Base_FVC"].astype(np.float32) + df["slope"] * (
        df["Weeks"].astype(np.float32) - anchor_week
    )

    # Change (score): constant confidence calibrated from anchored residuals; clipped to Kaggle rule.
    base_conf = float(np.clip(laplace_b, 70.0, 1000.0))
    conf = np.full(len(df), base_conf, dtype=np.float32)

    sub = pd.DataFrame(
        {
            "Patient_Week": df["Patient_Week"].values,
            "FVC": df["FVC_pred"].values.astype(np.float32),
            "Confidence": conf,
        }
    )
    return sub


if PYDICOM_OK and TF_OK and MODEL_OK:
    y_prediction_list = []
    for bi in range(len(pred_generator)):
        X1, X2 = pred_generator[bi]

        out_imgs = cnn_layer(tf.convert_to_tensor(X2))
        if isinstance(out_imgs, dict):
            out_imgs = list(out_imgs.values())[0]
        out_imgs = tf.convert_to_tensor(out_imgs)

        patients_in_batch = list(X1["Patient"].unique())
        counts = (
            X1["Patient"].value_counts().reindex(patients_in_batch).values.astype(int)
        )

        rep = []
        for i, cnt in enumerate(counts):
            rep.append(tf.repeat(tf.reshape(out_imgs[i], (1, -1)), cnt, axis=0))
        X_imgs = tf.concat(rep, axis=0)

        X_tab = tf.convert_to_tensor(
            np.asarray(X1[SELECTED_COLUMNS]).astype(np.float32)
        )

        pred = model_layer([X_imgs, X_tab])
        if isinstance(pred, dict):
            pred = list(pred.values())[0]
        pred = tf.convert_to_tensor(pred).numpy()

        y_prediction_list.append(pred)

    y_prediction = np.concatenate(y_prediction_list, axis=0)

    if len(y_prediction) != len(X_prediction):
        raise RuntimeError(
            f"Prediction length mismatch: got {len(y_prediction)} preds for {len(X_prediction)} rows"
        )

    fvc_pred = y_prediction[:, 1].astype(np.float32)
    conf_pred = (y_prediction[:, 2] - y_prediction[:, 0]).astype(np.float32)
    conf_pred = np.clip(conf_pred, 70.0, 1000.0)

    sub = pd.DataFrame(
        {
            "Patient_Week": X_prediction["Patient_Week"].values,
            "FVC": fvc_pred,
            "Confidence": conf_pred,
        }
    )
else:
    sub = _build_tabular_fallback_submission(X_prediction)

sample = pd.read_csv(
    "/kaggle/input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)
sub = sample[["Patient_Week"]].merge(sub, on="Patient_Week", how="left")
sub["FVC"] = sub["FVC"].fillna(sample["FVC"]).astype(np.float32)
sub["Confidence"] = sub["Confidence"].fillna(250.0).astype(np.float32)
sub["Confidence"] = np.clip(sub["Confidence"].values, 70.0, 1000.0)

sub.to_csv("submission.csv", index=False)
print(sub.head())
print("Wrote submission.csv with shape:", sub.shape)
print("Columns:", list(sub.columns))
assert list(sub.columns) == ["Patient_Week", "FVC", "Confidence"]
assert str("submission.csv").endswith(".csv")

Global slope (ml/week): -3.63413667678833
Estimated Laplace b = mean(|residual|) (ml): 118.9197006225586
                   Patient_Week         FVC  Confidence
0  ID00126637202218610655908_-3  2437.09375  118.919701
1  ID00126637202218610655908_-3  2437.09375  118.919701
2  ID00126637202218610655908_-3  2437.09375  118.919701
3  ID00126637202218610655908_-3  2437.09375  118.919701
4  ID00126637202218610655908_-3  2437.09375  118.919701
Wrote submission.csv with shape: (202248, 3)
Columns: ['Patient_Week', 'FVC', 'Confidence']
